# linalg-solve-batched — worked example 2: Batch-solve 2D ray–segment intersections as a system of linear equations

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `linalg-solve-batched`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

Ray–segment intersection can be framed as solving a 2×2 linear system: given a ray origin `O` and direction `D`, and segment endpoints `P0`, `P1`, find parameters `(t, s)` such that `O + t*D = P0 + s*(P1-P0)`. Rearranging gives a 2×2 system per ray–segment pair, which `torch.linalg.solve` handles in a single batched call.

## Worked solution

Step 1: For each ray `i`, the system is `[D, -(P1-P0)] @ [t, s]^T = P0 - O`. The matrix is `A[i] = [[Dx, -(P1x-P0x)], [Dy, -(P1y-P0y)]]` and the RHS is `b[i] = [P0x - Ox, P0y - Oy]`.

Step 2: Stack `A` as shape `(K, 2, 2)` and `b` as shape `(K, 2)`.

Step 3: Call `params = t.linalg.solve(A, b)`. The first column `params[:, 0]` gives the ray parameter `t`; `params[:, 1]` gives the segment parameter `s`.

Step 4: A valid intersection has `t >= 0` (in front of origin) and `0 <= s <= 1` (within segment). Print which rays intersect.

In [ ]:
import torch as t

t.manual_seed(7)

# K parallel ray–segment intersection queries
K = 6

# Rays: origins O and directions D (all pointing right-ish)
O = t.zeros(K, 2)      # all start at origin
D_vals = t.stack([
    t.tensor([1.0, 0.3]),
    t.tensor([1.0, -0.2]),
    t.tensor([1.0, 0.5]),
    t.tensor([0.8, 0.1]),
    t.tensor([1.0, 1.0]),
    t.tensor([1.0, -1.0]),
])

# Segments: vertical line at x=2, with various y-extents
P0 = t.tensor([[2.0, -0.5],  # seg0: y in [-0.5, 0.5] -> intersects ray0,1
               [2.0, -0.5],
               [2.0,  0.1],
               [2.0, -0.1],
               [2.0,  0.8],
               [2.0, -0.9]])
P1 = t.tensor([[2.0,  0.5],
               [2.0,  0.5],
               [2.0,  0.6],
               [2.0,  0.2],
               [2.0,  1.2],
               [2.0, -1.1]])

# Build A: (K, 2, 2) and b: (K, 2)
delta = P1 - P0                               # (K, 2)
# Columns of A: D and -delta
A = t.stack([D_vals, -delta], dim=-1)         # (K, 2, 2) — each column is a vector
b = P0 - O                                    # (K, 2)

params = t.linalg.solve(A, b)                 # (K, 2)
ray_t = params[:, 0]    # ray parameter
seg_s = params[:, 1]    # segment parameter

print(f'params shape: {params.shape}')        # (6, 2)
for i in range(K):
    valid = ray_t[i].item() >= 0 and 0 <= seg_s[i].item() <= 1
    print(f'  ray {i}: t={ray_t[i].item():.3f}  s={seg_s[i].item():.3f}  intersects={valid}')